In [10]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from models.CNN import SoundCNN
from trainer import Train
import config as cfg
from preprocessing import AudioPreprocessor
from dataloader import Dataloader

DATA_PATH = cfg.PROJECT_ROOT / "datasets"
FOLDS = ["fold1", "fold3", "fold4", "fold5", "fold6", "fold7", "fold8", "fold9"]

VAL_FOLD = "fold2"
TEST_FOLD = "fold10"

# Create model instance with SE and Attention blocks
model_instance = SoundCNN(num_classes=10, SqueezeExcitation=True, AttentionBlock=True, in_channels=5)

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [11]:
# Train on ALL folds without cross-validation for DeepFool testing
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import os
import copy

# Create a fresh model instance
model_all_folds = SoundCNN(num_classes=10, SqueezeExcitation=True, AttentionBlock=True, in_channels=5)

# Training setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_all_folds.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_all_folds.parameters(), lr=1e-3, weight_decay=1e-4)

train_loader_all = DataLoader(
    Dataloader(dataset_path=DATA_PATH, folds=FOLDS, include_augmented=False, use_cache=True),
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    Dataloader(dataset_path=DATA_PATH, folds=[VAL_FOLD], use_cache=True),
    batch_size=64,
    shuffle=False
)

def validate(model, dataset_type, dataloader):
    
    model.to(device)
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    
    progress_bar = tqdm(dataloader, desc="Validating", leave=True)
    with torch.no_grad():
        for inputs, folds, labels in progress_bar:
            if dataset_type == "singlechannel" or dataset_type == "augmentation_singlechannel":
                inputs = inputs[:, 0, ...].unsqueeze(1)  # for singlechannel
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            
            # Atualizar barra de progresso
            current_acc = correct / total if total > 0 else 0.0
            current_loss = running_loss / total if total > 0 else 0.0
            progress_bar.set_postfix({'loss': f'{current_loss:.4f}', 'acc': f'{current_acc:.4f}'})
            
    avg_loss = running_loss / total if total > 0 else 0.0
    accuracy = correct / total if total > 0 else 0.0
    return avg_loss, accuracy

print(f"Training on device: {device}")
print(f"Training on folds 1 to 8")
print(f"Total samples: {len(train_loader_all.dataset)}")

# Training parameters
epochs = 50
patience = 5
best_loss = float('inf')
epochs_no_improve = 0
best_model_wts = None

# Training loop
for epoch in range(1, epochs + 1):
    model_all_folds.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    progress_bar = tqdm(train_loader_all, desc=f"Epoch {epoch}/{epochs}")
    for inputs, folds, labels in progress_bar:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model_all_folds(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
        current_acc = correct / total
        current_loss = running_loss / total
        progress_bar.set_postfix({'loss': f'{current_loss:.4f}', 'acc': f'{current_acc:.4f}'})
    
    avg_loss = running_loss / total
    accuracy = correct / total
    
    print(f"Epoch {epoch}/{epochs} - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")
    
    val_loss, val_acc = validate(model_all_folds, "multichannel", val_loader)
    # Early stopping based on training loss
    if val_loss < best_loss:
        best_loss = val_loss
        epochs_no_improve = 0
        best_model_wts = copy.deepcopy(model_all_folds.state_dict())
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch} epochs")
            break

# Load best weights
if best_model_wts is not None:
    model_all_folds.load_state_dict(best_model_wts)

# Save the model
save_dir = os.path.join(os.getcwd(), "models_for_adversarial")
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, "ASECNN_all_folds.pth")
torch.save(model_all_folds.state_dict(), model_path)

print(f"\n✓ Model trained on all folds saved to: {model_path}")
print(f"Final training loss: {best_loss:.4f}")
print(f"Final training accuracy: {accuracy:.4f}")

Training on device: cuda
Training on folds 1 to 8
Total samples: 7007


Epoch 1/50: 100%|██████████| 110/110 [00:07<00:00, 14.12it/s, loss=1.7333, acc=0.3751]


Epoch 1/50 - Loss: 1.7333, Accuracy: 0.3751


Epoch 2/50: 100%|██████████| 110/110 [00:07<00:00, 14.45it/s, loss=1.2017, acc=0.5723]


Epoch 2/50 - Loss: 1.2017, Accuracy: 0.5723


...

Epoch 11/50 - Loss: 0.4311, Accuracy: 0.8634


Epoch 12/50: 100%|██████████| 110/110 [00:07<00:00, 14.85it/s, loss=0.3578, acc=0.8874]


Epoch 12/50 - Loss: 0.3578, Accuracy: 0.8874


Validating: 100%|██████████| 14/14 [00:00<00:00, 35.25it/s, loss=1.1661, acc=0.6475]

Early stopping triggered after 12 epochs

✓ Model trained on all folds saved to: c:\deep-learning-urban-sound-data\deepfool\models_for_adversarial\ASECNN_all_folds.pth
Final training loss: 0.9034
Final training accuracy: 0.8874


In [ ]:
# Train singlechannel model on ALL folds without cross-validation
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import os
import copy

# Create a fresh singlechannel model instance
singlechannel_model = SoundCNN(num_classes=10, SqueezeExcitation=False, in_channels=1)

# Training setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
singlechannel_model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(singlechannel_model.parameters(), lr=1e-3, weight_decay=1e-4)

train_loader_all = DataLoader(
    Dataloader(dataset_path=DATA_PATH, folds=FOLDS, include_augmented=False, use_cache=True),
    batch_size=128,
    shuffle=True
)

print(f"Training singlechannel model on device: {device}")
print(f"Training on all {len(all_folds)} folds")
print(f"Total samples: {len(train_loader_all.dataset)}")

# Training parameters
epochs = 50
patience = 10
best_loss = float('inf')
epochs_no_improve = 0
best_model_wts = None

# Training loop
for epoch in range(1, epochs + 1):
    singlechannel_model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    progress_bar = tqdm(train_loader_all, desc=f"Epoch {epoch}/{epochs}")
    for inputs, folds, labels in progress_bar:
        # Select only first channel for singlechannel model
        inputs = inputs[:, 0, ...].unsqueeze(1)
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = singlechannel_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
        current_acc = correct / total
        current_loss = running_loss / total
        progress_bar.set_postfix({'loss': f'{current_loss:.4f}', 'acc': f'{current_acc:.4f}'})
    
    avg_loss = running_loss / total
    accuracy = correct / total
    
    print(f"Epoch {epoch}/{epochs} - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")
    
    # Early stopping based on training loss
    if avg_loss < best_loss:
        best_loss = avg_loss
        epochs_no_improve = 0
        best_model_wts = copy.deepcopy(singlechannel_model.state_dict())
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch} epochs")
            break

# Load best weights
if best_model_wts is not None:
    singlechannel_model.load_state_dict(best_model_wts)

# Save the model
save_dir = os.path.join(os.getcwd(), "models_for_adversarial")
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, "CNN_singlechannel_all_folds.pth")
torch.save(singlechannel_model.state_dict(), model_path)

print(f"\n✓ Singlechannel model trained on all folds saved to: {model_path}")
print(f"Final training loss: {best_loss:.4f}")
print(f"Final training accuracy: {accuracy:.4f}")

Training singlechannel model on device: cuda
Training on all 10 folds
Total samples: 7079


Epoch 1/50: 100%|██████████| 56/56 [00:04<00:00, 11.35it/s, loss=1.8077, acc=0.3431]


Epoch 1/50 - Loss: 1.8077, Accuracy: 0.3431


Epoch 2/50: 100%|██████████| 56/56 [00:04<00:00, 11.46it/s, loss=1.3681, acc=0.5042]


Epoch 2/50 - Loss: 1.3681, Accuracy: 0.5042


...

Epoch 48/50 - Loss: 0.1996, Accuracy: 0.9370


Epoch 49/50: 100%|██████████| 56/56 [00:04<00:00, 11.94it/s, loss=0.2132, acc=0.9342]


Epoch 49/50 - Loss: 0.2132, Accuracy: 0.9342


Epoch 50/50: 100%|██████████| 56/56 [00:04<00:00, 11.92it/s, loss=0.2236, acc=0.9292]

Epoch 50/50 - Loss: 0.2236, Accuracy: 0.9292

✓ Singlechannel model trained on all folds saved to: c:\deep-learning-urban-sound-data\models_for_adversarial\CNN_singlechannel_all_folds.pth
Final training loss: 0.1996
Final training accuracy: 0.9292
